# 使用 `torch.nn` 完成一个分类训练案例

这个案例使用 `nn.Sequential` 构建多层感知机，对二维数据进行三分类。完整流程包括：准备数据、定义模型、训练、评估和预测。

In [5]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

## 1. 准备数据

生成三个以不同位置为中心的数据簇。每个样本有两个特征，标签分别为 `0`、`1`、`2`。

In [ ]:
def make_classification_data(samples_per_class=200):
    centers = torch.tensor([[-2.0, -1.5], [2.0, -1.0], [0.0, 2.0]])
    features = []
    labels = []

    for class_id, center in enumerate(centers):
        features.append(center + 0.65 * torch.randn(samples_per_class, 2))
        labels.append(torch.full((samples_per_class,), class_id, dtype=torch.long))


    X = torch.cat(features)
    y = torch.cat(labels)
    indices = torch.randperm(len(X))
    return X[indices], y[indices]


X, y = make_classification_data()
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=32,
    shuffle=True,
)

print(f"训练集: {X_train.shape}, 测试集: {X_test.shape}")
print(f"一个批次: {next(iter(train_loader))[0].shape}")

Features:
[tensor([[-1.4961, -1.5691],
        [-1.4210, -2.0026],
        [-1.1390, -2.6515],
        [-1.9370, -1.2612],
        [-1.5284, -0.7954],
        [-2.2340, -1.5166],
        [-2.0997, -0.7934],
        [-1.5579, -0.7449],
        [-0.7502, -1.6445],
        [-1.2660, -2.0420],
        [-2.8578, -2.3428],
        [-1.7875, -1.1058],
        [-1.0721, -1.2122],
        [-2.8219, -2.2304],
        [-1.6186, -0.7402],
        [-1.4133, -2.6159],
        [-2.0189, -0.7537],
        [-2.7532, -1.3935],
        [-2.3564, -1.1459],
        [-1.0105, -1.4847],
        [-1.2524, -1.4748],
        [-2.4965, -2.8061],
        [-2.3322, -1.9585],
        [-1.2496, -2.6351],
        [-1.8355, -2.0136],
        [-1.1216, -1.5470],
        [-0.6441, -1.1030],
        [-1.8607, -1.6136],
        [-2.3874, -2.5105],
        [-1.0179, -2.4515],
        [-2.3828, -1.4826],
        [-2.1626, -1.5594],
        [-2.6807, -0.5195],
        [-2.9117, -2.3704],
        [-3.3768, -0.9435],
        [

## 2. 使用 `nn` 定义模型

`nn.Linear` 执行线性变换，`nn.ReLU` 引入非线性。最后一层输出三个原始分数（logits），不需要手动添加 `Softmax`，因为 `nn.CrossEntropyLoss` 内部已经包含相应计算。

In [ ]:
model = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 16),
    nn.ReLU(),
    nn.Linear(16, 3),
)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

sample_logits = model(X_train[:5])
assert sample_logits.shape == (5, 3)
print(model)
print("输出形状:", sample_logits.shape)

## 3. 训练模型

每个批次都执行四步：前向传播、计算损失、反向传播、更新参数。调用 `optimizer.zero_grad()` 是为了清除上一个批次累积的梯度。

In [ ]:
def train_one_epoch(model, data_loader, loss_fn, optimizer):
    model.train()
    total_loss = 0.0

    for batch_X, batch_y in data_loader:
        logits = model(batch_X)
        loss = loss_fn(logits, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(batch_X)

    return total_loss / len(data_loader.dataset)


loss_history = []
for epoch in range(1, 51):
    epoch_loss = train_one_epoch(model, train_loader, loss_fn, optimizer)
    loss_history.append(epoch_loss)
    if epoch == 1 or epoch % 10 == 0:
        print(f"epoch {epoch:02d} | loss {epoch_loss:.4f}")

assert loss_history[-1] < loss_history[0], "训练后损失应低于初始损失"

## 4. 评估模型

评估时使用 `model.eval()` 切换到推理模式，并通过 `torch.inference_mode()` 关闭梯度记录。分类结果取 logits 最大值所在的位置。

In [ ]:
def accuracy(model, X, y):
    model.eval()
    with torch.inference_mode():
        predictions = model(X).argmax(dim=1)
    return (predictions == y).float().mean().item()


test_accuracy = accuracy(model, X_test, y_test)
print(f"测试集准确率: {test_accuracy:.2%}")
assert test_accuracy > 0.90, "该固定数据集上的准确率应超过 90%"

## 5. 对新样本进行预测

In [ ]:
new_samples = torch.tensor([[-2.0, -1.5], [2.0, -1.0], [0.0, 2.0]])

model.eval()
with torch.inference_mode():
    logits = model(new_samples)
    probabilities = logits.softmax(dim=1)
    predicted_classes = probabilities.argmax(dim=1)

for sample, predicted_class, probability in zip(
    new_samples, predicted_classes, probabilities.max(dim=1).values
):
    print(
        f"样本 {sample.tolist()} -> 类别 {predicted_class.item()} "
        f"(置信度 {probability.item():.2%})"
    )